In [1]:
#Download the dataset
import urllib.request
url = "https://huggingface.co/datasets/BigDataUB/synthetic-columnar/resolve/main/orders.csv"
print("Downloading dataset (~830MB, this will take a few minutes)...")
urllib.request.urlretrieve(url, "/app/data/orders.csv")
print("Done!")

Done!


Exercise 1: Explore the CSV

In [2]:
import os
import time
import pyarrow as pa
import pyarrow.csv as pcsv
import pyarrow.parquet as pq
import duckdb
import pandas as pd

In [3]:
pa.__version__

'23.0.1'

In [6]:
table = pa.csv.read_csv("/app/data/orders.csv")

In [7]:
table.schema

order_id: int64
order_date: date32[day]
customer_id: int64
country: string
category: string
product: string
quantity: int64
unit_price: double
total_amount: double
payment_method: string
status: string

In [8]:
table.num_rows

10000000

In [9]:
table.num_columns

11

In [10]:
os.path.getsize("/app/data/orders.csv")

829423674

In [17]:
start = time.time()
table = pa.csv.read_csv("/app/data/orders.csv")
end = time.time()
print(f"Time to open the csv: {end-start}")


Time to open the csv: 2.48442006111145


Exercise 2: Format conversion - CSV → JSON → Parquet

In [15]:
df = table.slice(0, 100_000).to_pandas()
df.to_json("/app/data/orders_sample.json", orient="records", lines=True)

In [19]:
start = time.time()
pq.write_table(table, where="/app/data/orders_sample.parquet")
end = time.time()
print(f"Parquet write time: {(end-start):.2f}s")

Parquet write time: 7.53s


In [21]:
metadata = pq.read_metadata("/app/data/orders_sample.parquet")
print(metadata)

  created_by: parquet-cpp-arrow version 23.0.1
  num_columns: 11
  num_rows: 10000000
  num_row_groups: 10
  format_version: 2.6
  serialized_size: 14195


In [22]:
print(metadata.row_group(0).column(0))


  file_offset: 0
  file_path: 
  physical_type: INT64
  num_values: 1048576
  path_in_schema: order_id
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 1
      max: 1048576
      null_count: 0
      distinct_count: None
      num_values: 1048576
      physical_type: INT64
      logical_type: None
      converted_type (legacy): NONE
  geo_statistics:
    None
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
  has_dictionary_page: True
  dictionary_page_offset: 4
  data_page_offset: 525089
  total_compressed_size: 4471417
  total_uncompressed_size: 8663180


In [24]:
# Q7. Compare file sizes. Fill in the table:

#CSV
print(os.path.getsize("/app/data/orders.csv"))

#JSON
print(os.path.getsize("/app/data/orders_sample.json"))

#PARQUET
print(os.path.getsize("/app/data/orders_sample.parquet"))

829423674
22893877
185991626


In [25]:
for name, path in [("CSV", "orders.csv"), ("JSON (100k)", "orders_sample.json"),
                    ("Parquet (default)", "orders_default.parquet")]:
    size = os.path.getsize(f"/app/data/{path}")
    print(f"{name}: {size / 1e6:.1f} MB")

CSV: 829.4 MB
JSON (100k): 22.9 MB


FileNotFoundError: [Errno 2] No such file or directory: '/app/data/orders_default.parquet'

In [ ]:
# Exercise 3: Compression comparison

print("")